## 🔐 Prerequisites

Before running the first cell, make sure you're authenticated with Azure CLI:

```bash
az login
```

# 📈 Azure AI Agent Observability

## Industry Use Case: Customer Service Monitoring

This notebook demonstrates observability for **Azure AI agents** using the AzureAIClient.

| Feature | FSI Application |
|---------|-----------------|
| **Automatic Tracing** | Monitor customer service interactions |
| **Span Context** | Track conversation flow |
| **Application Insights** | Centralized logging |

In [ ]:
# Copyright (c) Microsoft. All rights reserved.
import os
import sys
from importlib.metadata import version
from pathlib import Path

from dotenv import load_dotenv

assert version("agent-framework-core") == "1.17.0", "Select the pinned project kernel."
repo_root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "requirements.in").is_file())
assert Path(sys.executable).resolve() == (repo_root / ".venv/Scripts/python.exe").resolve(), (
    "Select the repository .venv kernel."
)
load_dotenv(repo_root / ".env", override=False)

PROJECT_ENDPOINT = (
    os.getenv("FOUNDRY_PROJECT_ENDPOINT")
    or os.getenv("AI_FOUNDRY_PROJECT_ENDPOINT")
    or os.getenv("AZURE_AI_PROJECT_ENDPOINT")
)
MODEL_DEPLOYMENT = os.getenv("FOUNDRY_MODEL") or os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")
if not PROJECT_ENDPOINT or not MODEL_DEPLOYMENT:
    raise ValueError("Set a Foundry project endpoint and model deployment name in the environment.")

print(f"Kernel: {sys.executable}\nPython: {sys.version.split()[0]}")
print("✅ Environment loaded (project endpoint hidden)")


In [ ]:
import asyncio
from random import uniform

from agent_framework import Agent
from agent_framework.foundry import FoundryChatClient
from agent_framework.observability import get_tracer
from azure.ai.projects.aio import AIProjectClient
from azure.identity import AzureCliCredential
from azure.monitor.opentelemetry import configure_azure_monitor
from opentelemetry.trace import SpanKind
from opentelemetry.trace.span import format_trace_id

print("✅ All imports loaded")

## Define Customer Service Tools

In [ ]:
async def get_account_info(account_id: str) -> str:
    """Get customer account information."""
    await asyncio.sleep(uniform(0.1, 0.3))  # Simulate DB lookup
    accounts = {
        "ACC-001": {"name": "John Smith", "balance": 15432.50, "status": "Active"},
        "ACC-002": {"name": "Jane Doe", "balance": 8750.25, "status": "Active"},
    }
    info = accounts.get(account_id, {"name": "Unknown", "balance": 0, "status": "Not Found"})
    return f"Account {account_id}: {info['name']}, Balance: ${info['balance']:,.2f}, Status: {info['status']}"


async def create_support_ticket(issue: str, priority: str = "medium") -> str:
    """Create a customer support ticket."""
    await asyncio.sleep(uniform(0.1, 0.2))  # Simulate API call
    ticket_id = f"TKT-{hash(issue) % 10000:04d}"
    return f"Support ticket {ticket_id} created with {priority} priority: {issue[:50]}..."


print("✅ Tools defined: get_account_info, create_support_ticket")

## Run Agent with Observability

The `AzureAIClient` provides automatic telemetry configuration via `configure_azure_monitor()`.

In [ ]:
async def run_customer_service_agent():
    """Run customer service agent with observability."""
    print("\n--- Customer Service Agent with Observability ---\n")

    with AzureCliCredential() as credential:
        async with AIProjectClient(
            endpoint=PROJECT_ENDPOINT,
            credential=credential,
        ) as project_client:
            # Configure Azure Monitor with App Insights from Foundry
            try:
                app_insights_conn = await project_client.telemetry.get_application_insights_connection_string()
                if app_insights_conn:
                    configure_azure_monitor(connection_string=app_insights_conn)
                    print("✅ Azure Monitor configured with App Insights\n")
                else:
                    print("⚠️ No Application Insights configured in Foundry project")
                    print("Continuing without Azure Monitor telemetry...\n")
            except Exception as e:
                print(f"⚠️ Could not configure Azure Monitor: {e}")
                print("Continuing without telemetry...\n")

            # Create agent using FoundryChatClient + Agent
            client = FoundryChatClient(
                project_endpoint=PROJECT_ENDPOINT,
                model=MODEL_DEPLOYMENT,
                credential=credential,
            )
            try:
                agent = Agent(
                    client=client,
                    name="CustomerServiceAgent",
                    instructions="You are a helpful customer service agent for a bank. Help customers with account inquiries and create support tickets when needed.",
                    tools=[get_account_info, create_support_ticket],
                )

                # Customer inquiries
                inquiries = [
                    "What's the balance on account ACC-001?",
                    "I can't access my online banking. Can you create a ticket?",
                    "Check account ACC-002 and create a high priority ticket for unauthorized transaction",
                ]

                # Run with tracing span
                with get_tracer().start_as_current_span("Customer Service Session", kind=SpanKind.CLIENT) as span:
                    trace_id = format_trace_id(span.get_span_context().trace_id)
                    print(f"🔍 Trace ID: {trace_id}")
                    print("(View in Azure Portal > Application Insights)\n")

                    # A session keeps conversation state across the streamed inquiries.
                    session = agent.create_session()
                    for inquiry in inquiries:
                        print(f"Customer: {inquiry}")
                        print(f"{agent.name}: ", end="")
                        async for update in agent.run(inquiry, session=session, stream=True):
                            if update.text:
                                print(update.text, end="")
                        print("\n")
            finally:
                await client.client.close()
                await client.project_client.close()

await run_customer_service_agent()


## Key Takeaways

| Feature | Description |
|---------|-------------|
| `FoundryChatClient` + `Agent` | Application-owned agent for Azure AI (Foundry) |
| `configure_azure_monitor()` | Telemetry setup with App Insights connection string |
| `get_tracer()` | Get current OpenTelemetry tracer |
| `start_as_current_span()` | Create custom trace spans |
| `agent.create_session()` | Keep conversation state across streamed turns |
| `agent.run(..., session=..., stream=True)` | Stream responses within a session |

## Telemetry Setup Pattern

1. Read the App Insights connection string from the Foundry project via `project_client.telemetry.get_application_insights_connection_string()`.
2. Call `configure_azure_monitor(connection_string=...)` once.
3. Wrap agent runs in a custom span with `get_tracer().start_as_current_span(...)`.
4. View traces in **Azure Portal > Application Insights > Transaction Search**.
